# Session 3, Module 03: File I/O


This module covers:
- Reading and writing text files
- Working with CSV files
- Working with JSON files
- Using pathlib for path operations
- Binary files basics

Data Engineering Context:
File I/O is fundamental for ETL pipelines - reading source data,
writing transformed results, and managing configuration files.


In [18]:
import csv
import json
from pathlib import Path
import os

## Pathlib — Modern Path Handling


In [2]:
print("=== Pathlib — Modern Path Handling ===")

=== Pathlib — Modern Path Handling ===


pathlib provides an object-oriented interface for paths
Much cleaner than os.path string manipulation
Creating paths

In [3]:
current_dir = Path(".")
home_dir = Path.home()
data_dir = Path("/data/warehouse")

print(f"Current directory: {current_dir.absolute()}")
print(f"Home directory: {home_dir}")
print(f"Data directory: {data_dir}")

# Path operations
file_path = Path("/data/warehouse/customers/2024/01/data.csv")

print(f"\nPath components:")
print(f"  Name: {file_path.name}")          # OUTPUT: data.csv
print(f"  Stem: {file_path.stem}")          # OUTPUT: data
print(f"  Suffix: {file_path.suffix}")      # OUTPUT: .csv
print(f"  Parent: {file_path.parent}")      # OUTPUT: /data/warehouse/customers/2024/01
print(f"  Parts: {file_path.parts}")        # OUTPUT: ('/', 'data', 'warehouse', ...)

# Building paths with / operator (very Pythonic!)
base = Path("/data/warehouse")
full_path = base / "customers" / "2024" / "data.csv"
print(f"\nBuilt path: {full_path}")  # OUTPUT: /data/warehouse/customers/2024/data.csv

# Checking path properties
script_path = Path(__file__) if "__file__" in dir() else Path(".")
print(f"\nPath checks:")
print(f"  exists(): {script_path.exists()}")
print(f"  is_file(): {script_path.is_file()}")
print(f"  is_dir(): {script_path.is_dir()}")

Current directory: /Users/ahadmammadli/Desktop/Ingreess_DE/bootcamp-topics/depython/session_3_professional_python
Home directory: /Users/ahadmammadli
Data directory: /data/warehouse

Path components:
  Name: data.csv
  Stem: data
  Suffix: .csv
  Parent: /data/warehouse/customers/2024/01
  Parts: ('/', 'data', 'warehouse', 'customers', '2024', '01', 'data.csv')

Built path: /data/warehouse/customers/2024/data.csv

Path checks:
  exists(): True
  is_file(): False
  is_dir(): True


## Reading Text Files


In [4]:
print("\n=== Reading Text Files ===")

# Create a demo directory in the current working directory
temp_dir = Path(".") / "depython_demo"
temp_dir.mkdir(exist_ok=True)
sample_file = temp_dir / "sample.txt"

# Write sample content
sample_file.write_text("""Line 1: Hello, World!
Line 2: Data Engineering
Line 3: Python is great
Line 4: ETL pipelines
Line 5: The end""")

# Method 1: read_text() — reads entire file
content = sample_file.read_text()
print(f"Full content ({len(content)} chars):")
print(content[:50] + "...")

# Method 2: open() with context manager — for line-by-line processing
print("\nLine-by-line reading:")
with open(sample_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        print(f"  {i}: {line.strip()}")
        if i >= 3:
            print("  ...")
            break

# Method 3: readlines() — get list of lines
with open(sample_file, "r") as f:
    lines = f.readlines()
print(f"\nTotal lines: {len(lines)}")

# Method 4: Read with Path.open()
with sample_file.open("r") as f:
    first_line = f.readline()
print(f"First line: {first_line.strip()}")


=== Reading Text Files ===
Full content (108 chars):
Line 1: Hello, World!
Line 2: Data Engineering
Lin...

Line-by-line reading:
  1: Line 1: Hello, World!
  2: Line 2: Data Engineering
  3: Line 3: Python is great
  ...

Total lines: 5
First line: Line 1: Hello, World!


## Writing Text Files


In [5]:
print("\n=== Writing Text Files ===")

output_file = temp_dir / "output.txt"

# Method 1: write_text() — write entire content at once
output_file.write_text("Simple one-line write\n")
print(f"Wrote to {output_file}")

# Method 2: open() with 'w' mode — write with more control
with open(output_file, "w", encoding="utf-8") as f:
    f.write("Line 1\n")
    f.write("Line 2\n")
    f.writelines(["Line 3\n", "Line 4\n"])  # Write multiple lines

# Method 3: Append with 'a' mode
with open(output_file, "a", encoding="utf-8") as f:
    f.write("Appended line\n")

print(f"Content after writes:")
print(output_file.read_text())


=== Writing Text Files ===
Wrote to depython_demo/output.txt
Content after writes:
Line 1
Line 2
Line 3
Line 4
Appended line



## Working With Csv Files


In [7]:
print("\n=== Working with CSV Files ===")

csv_file = temp_dir / "customers.csv"

# Writing CSV — DictWriter is most common in data engineering
customers = [
    {"id": 1, "name": "Alice Smith", "email": "alice@example.com", "status": "active"},
    {"id": 2, "name": "Bob Jones", "email": "bob@example.com", "status": "inactive"},
    {"id": 3, "name": "Carol White", "email": "carol@example.com", "status": "active"},
]

# Write CSV with DictWriter
with open(csv_file, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["id", "name", "email", "status"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)

    writer.writeheader()  # Write column names
    writer.writerows(customers)  # Write all rows

print(f"Wrote CSV to {csv_file}")
print(f"Content:\n{csv_file.read_text()}")

# Reading CSV — DictReader gives you dictionaries
print("Reading CSV with DictReader:")
with open(csv_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"  {row['id']}: {row['name']} ({row['status']})")

# Reading CSV as raw rows (list of lists)
print("\nReading CSV with reader (raw rows):")
with open(csv_file, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)  # First row is header
    print(f"  Header: {header}")
    for row in reader:
        print(f"  Row: {row}")


=== Working with CSV Files ===
Wrote CSV to depython_demo/customers.csv
Content:
id,name,email,status
1,Alice Smith,alice@example.com,active
2,Bob Jones,bob@example.com,inactive
3,Carol White,carol@example.com,active

Reading CSV with DictReader:
  1: Alice Smith (active)
  2: Bob Jones (inactive)
  3: Carol White (active)

Reading CSV with reader (raw rows):
  Header: ['id', 'name', 'email', 'status']
  Row: ['1', 'Alice Smith', 'alice@example.com', 'active']
  Row: ['2', 'Bob Jones', 'bob@example.com', 'inactive']
  Row: ['3', 'Carol White', 'carol@example.com', 'active']


## Working With Json Files


In [8]:
print("\n=== Working with JSON Files ===")

json_file = temp_dir / "config.json"

# Python dict to write
config = {
    "pipeline_name": "daily_etl",
    "batch_size": 1000,
    "sources": ["customers", "orders", "products"],
    "settings": {
        "validate": True,
        "retry_count": 3,
        "timeout_seconds": 30
    }
}

# Writing JSON
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)  # indent for pretty printing

print(f"Wrote JSON to {json_file}")
print(f"Content:\n{json_file.read_text()}")

# Reading JSON
with open(json_file, "r", encoding="utf-8") as f:
    loaded_config = json.load(f)

print(f"\nLoaded config:")
print(f"  Pipeline: {loaded_config['pipeline_name']}")
print(f"  Batch size: {loaded_config['batch_size']}")
print(f"  Sources: {loaded_config['sources']}")

# JSON with custom objects (using default parameter)
from datetime import datetime

data_with_dates = {
    "name": "report",
    "generated_at": datetime.now(),
    "records": 1000
}


=== Working with JSON Files ===
Wrote JSON to depython_demo/config.json
Content:
{
  "pipeline_name": "daily_etl",
  "batch_size": 1000,
  "sources": [
    "customers",
    "orders",
    "products"
  ],
  "settings": {
    "validate": true,
    "retry_count": 3,
    "timeout_seconds": 30
  }
}

Loaded config:
  Pipeline: daily_etl
  Batch size: 1000
  Sources: ['customers', 'orders', 'products']


datetime is not JSON serializable by default
Use default parameter to handle custom types

In [9]:
def json_serializer(obj):
    """Custom JSON serializer for non-standard types."""
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")


json_with_dates = json.dumps(data_with_dates, default=json_serializer, indent=2)
print(f"\nJSON with datetime:\n{json_with_dates}")


JSON with datetime:
{
  "name": "report",
  "generated_at": "2026-06-16T16:54:17.272552",
  "records": 1000
}


## Json Lines Format (Jsonl)


In [10]:
print("\n=== JSON Lines Format (JSONL) ===")


=== JSON Lines Format (JSONL) ===


JSONL: One JSON object per line
Common in data engineering for streaming/append scenarios

In [11]:
jsonl_file = temp_dir / "events.jsonl"

events = [
    {"event": "click", "user_id": 1, "timestamp": "2024-01-15T10:30:00"},
    {"event": "purchase", "user_id": 1, "amount": 99.99},
    {"event": "click", "user_id": 2, "timestamp": "2024-01-15T10:35:00"},
]

# Writing JSONL
with open(jsonl_file, "w", encoding="utf-8") as f:
    for event in events:
        f.write(json.dumps(event) + "\n")

print(f"Wrote JSONL to {jsonl_file}")
print(f"Content:\n{jsonl_file.read_text()}")

# Reading JSONL
print("Reading JSONL:")
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        event = json.loads(line.strip())
        print(f"  {event['event']}: user {event['user_id']}")

Wrote JSONL to depython_demo/events.jsonl
Content:
{"event": "click", "user_id": 1, "timestamp": "2024-01-15T10:30:00"}
{"event": "purchase", "user_id": 1, "amount": 99.99}
{"event": "click", "user_id": 2, "timestamp": "2024-01-15T10:35:00"}

Reading JSONL:
  click: user 1
  purchase: user 1
  click: user 2


## Directory Operations


In [12]:
print("\n=== Directory Operations ===")

# Create directory structure
project_dir = temp_dir / "etl_project"
(project_dir / "data" / "raw").mkdir(parents=True, exist_ok=True)
(project_dir / "data" / "processed").mkdir(parents=True, exist_ok=True)
(project_dir / "logs").mkdir(parents=True, exist_ok=True)

print(f"Created directory structure in {project_dir}")

# List directory contents
print("\nDirectory listing:")
for path in project_dir.rglob("*"):  # rglob for recursive
    rel_path = path.relative_to(project_dir)
    prefix = "  📁 " if path.is_dir() else "  📄 "
    print(f"{prefix}{rel_path}")

# Find files by pattern
print("\nFinding all .csv files:")
for csv_path in temp_dir.glob("**/*.csv"):
    print(f"  {csv_path.name}")


=== Directory Operations ===
Created directory structure in depython_demo/etl_project

Directory listing:
  📁 logs
  📁 data
  📁 data/processed
  📁 data/raw

Finding all .csv files:
  customers.csv


## File Encoding


In [13]:
print("\n=== File Encoding ===")


=== File Encoding ===


Always specify encoding explicitly!
UTF-8 is the standard for data engineering

In [14]:
unicode_file = temp_dir / "unicode.txt"
unicode_content = "Hello 世界 🌍 Привет"

# Write with UTF-8 encoding
unicode_file.write_text(unicode_content, encoding="utf-8")
print(f"Wrote: {unicode_content}")

# Read with UTF-8 encoding
read_content = unicode_file.read_text(encoding="utf-8")
print(f"Read: {read_content}")

Wrote: Hello 世界 🌍 Привет
Read: Hello 世界 🌍 Привет


Common encodings:
- utf-8: Universal, handles all characters (PREFERRED)
- latin-1: Western European, 8-bit
- cp1252: Windows Western European
- ascii: 7-bit, English only

## Binary Files


In [15]:
print("\n=== Binary Files ===")

binary_file = temp_dir / "data.bin"

# Writing binary data
binary_data = bytes([0x48, 0x65, 0x6C, 0x6C, 0x6F])  # "Hello" in bytes
with open(binary_file, "wb") as f:
    f.write(binary_data)

print(f"Wrote {len(binary_data)} bytes")

# Reading binary data
with open(binary_file, "rb") as f:
    read_binary = f.read()

print(f"Read binary: {read_binary}")
print(f"As string: {read_binary.decode('utf-8')}")


=== Binary Files ===
Wrote 5 bytes
Read binary: b'Hello'
As string: Hello


## Practical Example: Data File Processor


In [16]:
print("\n=== Practical Example: Data File Processor ===")


def process_data_files(input_dir: Path, output_dir: Path) -> dict:
    """
    Process all CSV files in input directory.

    Args:
        input_dir: Directory containing input CSV files
        output_dir: Directory for processed output files

    Returns:
        Statistics about processed files
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    stats = {
        "files_processed": 0,
        "total_records": 0,
        "errors": []
    }

    for csv_path in input_dir.glob("*.csv"):
        try:
            # Read input
            with open(csv_path, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                records = list(reader)

            # Process (simple example: add processed flag)
            for record in records:
                record["processed"] = "true"

            # Write output
            output_path = output_dir / f"processed_{csv_path.name}"
            if records:
                with open(output_path, "w", newline="", encoding="utf-8") as f:
                    writer = csv.DictWriter(f, fieldnames=records[0].keys())
                    writer.writeheader()
                    writer.writerows(records)

            stats["files_processed"] += 1
            stats["total_records"] += len(records)
            print(f"  Processed {csv_path.name}: {len(records)} records")

        except Exception as e:
            stats["errors"].append(f"{csv_path.name}: {str(e)}")

    return stats


# Run the processor
print("Processing data files:")
result = process_data_files(temp_dir, temp_dir / "processed")
print(f"\nResults: {result}")


=== Practical Example: Data File Processor ===
Processing data files:
  Processed customers.csv: 3 records

Results: {'files_processed': 1, 'total_records': 3, 'errors': []}


## Cleanup


In [17]:
print("\n=== Cleanup ===")
print(f"Demo files created in: {temp_dir}")
print("You can delete this directory manually if needed.")


=== Cleanup ===
Demo files created in: depython_demo
You can delete this directory manually if needed.


## Summary


In [17]:
print("\n=== Summary ===")
print("""
File I/O Key Points:

PATHLIB:
  - Use Path for all path operations
  - Build paths with / operator: base / "subdir" / "file.txt"
  - Methods: exists(), is_file(), is_dir(), glob(), rglob()

TEXT FILES:
  - Always specify encoding="utf-8"
  - Use context managers (with statement)
  - read_text() / write_text() for simple cases
  - open() for more control

CSV FILES:
  - DictReader/DictWriter for dict-based access
  - reader/writer for raw list access
  - Use newline="" when opening for csv module

JSON FILES:
  - json.load() / json.dump() for files
  - json.loads() / json.dumps() for strings
  - Use default parameter for custom types
  - JSONL for streaming: one object per line
""")


=== Summary ===

File I/O Key Points:

PATHLIB:
  - Use Path for all path operations
  - Build paths with / operator: base / "subdir" / "file.txt"
  - Methods: exists(), is_file(), is_dir(), glob(), rglob()

TEXT FILES:
  - Always specify encoding="utf-8"
  - Use context managers (with statement)
  - read_text() / write_text() for simple cases
  - open() for more control

CSV FILES:
  - DictReader/DictWriter for dict-based access
  - reader/writer for raw list access
  - Use newline="" when opening for csv module

JSON FILES:
  - json.load() / json.dump() for files
  - json.loads() / json.dumps() for strings
  - Use default parameter for custom types
  - JSONL for streaming: one object per line

